# Notebook 4: Merge and prep

Load the supplementary data from Notebook 3, group rare flat models, null-check everything, and save the analysis-ready dataset.

## Setup

In [1]:
%load_ext rpy2.ipython
%load_ext autoreload
%autoreload 2

%matplotlib inline
from matplotlib import rcParams
rcParams['figure.figsize'] = (16, 100)

import warnings
from rpy2.rinterface import RRuntimeWarning
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from IPython.display import display, HTML

Error importing in API mode: ImportError("dlopen(/Users/wongpeiting/.pyenv/versions/3.13.9/lib/python3.13/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib\n  Referenced from: <B96A8100-FA7A-3EFC-8726-931D26646DE6> /Users/wongpeiting/.pyenv/versions/3.13.9/lib/python3.13/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file)")


Trying to import in ABI mode.


In [2]:
%%R

require('tidyverse')
require('scales')

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.1     


── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Loading required package: tidyverse
Loading required package: scales

Attaching package: ‘scales’

The following object is masked from ‘package:purrr’:

    discard

The following object is masked from ‘package:readr’:

    col_factor



## Load cleaned data

In [3]:
df = pd.read_csv('data/hdb_resale_supplementary.csv', encoding='utf-8')
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nColumns: {df.columns.tolist()}")
df.head(3)

Loaded 50,718 rows x 61 columns

Columns: ['month', 'town', 'block', 'street_name', 'resale_price', 'log_resale_price', 'flat_type', 'flat_model', 'floor_area_sqm', 'storey_range', 'storey_mid', 'lease_commence_date', 'remaining_lease_years', 'remaining_lease', 'ends_in_8', 'last_digit', 'dataset', 'latitude', 'longitude', 'dist_cbd_km', 'mrt_dist_m', 'nearest_mrt_name', 'school_dist_m', 'nearest_school_name', 'hawker_dist_m', 'nearest_hawker_name', 'supermarket_dist_m', 'nearest_supermarket_name', 'park_dist_m', 'nearest_park_name', 'columbarium_dist_m', 'nearest_columbarium_name', 'funeral_dist_m', 'nearest_funeral_name', 'hospital_dist_m', 'nearest_hospital_name', 'temple_dist_m', 'nearest_temple_name', 'reservoir_dist_m', 'nearest_reservoir_name', 'coast_dist_m', 'nearest_coast_name', 'popular_school_dist_m', 'nearest_popular_school_name', 'num_eights_in_price', 'num_fours_in_price', 'price_has_888', 'price_has_444', 'price_has_168', 'price_has_138', 'price_has_328', 'num_eights_ta

,month,town,block,street_name,resale_price,log_resale_price,flat_type,flat_model,floor_area_sqm,storey_range,...,num_eights_tail,num_fours_tail,has_floor_4,has_floor_13,has_floor_14,block_has_4,block_has_8,block_num_eights,hungry_ghost,cny_month
0,2024-05-01,ANG MO KIO,405,ANG MO KIO AVE 10,350000.0,5.544068,3 ROOM,New Generation,67.0,10 TO 12,...,0,0,0,0,0,1,0,0,0,0
1,2024-05-01,ANG MO KIO,463,ANG MO KIO AVE 10,350000.0,5.544068,3 ROOM,New Generation,68.0,01 TO 03,...,0,0,0,0,0,1,0,0,0,0
2,2024-05-01,ANG MO KIO,542,ANG MO KIO AVE 10,352000.0,5.546543,3 ROOM,New Generation,68.0,01 TO 03,...,0,0,0,0,0,1,0,0,0,0


## Ensure correct types

CSV round-tripping can silently change types (integers to floats, dates to strings), so I explicitly cast each column here.

In [4]:
# Numeric columns
df['resale_price'] = pd.to_numeric(df['resale_price'], errors='coerce')
df['log_resale_price'] = pd.to_numeric(df['log_resale_price'], errors='coerce')
df['floor_area_sqm'] = pd.to_numeric(df['floor_area_sqm'], errors='coerce')
df['storey_mid'] = pd.to_numeric(df['storey_mid'], errors='coerce')
df['lease_commence_date'] = pd.to_numeric(df['lease_commence_date'], errors='coerce')
df['remaining_lease_years'] = pd.to_numeric(df['remaining_lease_years'], errors='coerce')
df['ends_in_8'] = pd.to_numeric(df['ends_in_8'], errors='coerce').astype('Int64')
df['last_digit'] = pd.to_numeric(df['last_digit'], errors='coerce').astype('Int64')

# String columns
for col in ['town', 'block', 'street_name', 'flat_type', 'flat_model', 'storey_range']:
    df[col] = df[col].astype(str)

print("Data types after casting:")
print(df.dtypes)

Data types after casting:
month                object
town                 object
block                object
street_name          object
resale_price        float64
                     ...   
block_has_4           int64
block_has_8           int64
block_num_eights      int64
hungry_ghost          int64
cny_month             int64
Length: 61, dtype: object


## Group rare flat models

Flat models with fewer than 50 transactions get grouped into "Other" -- too few data points for stable regression coefficients. The groupings are printed below so an editor can review.

In [5]:
THRESHOLD = 50

model_counts = df['flat_model'].value_counts()
print(f"=== flat_model counts (before grouping) ===")
print(model_counts)
print(f"\nTotal unique flat models: {len(model_counts)}")

# Identify rare models
rare_models = model_counts[model_counts < THRESHOLD].index.tolist()
print(f"\nModels with < {THRESHOLD} transactions (will be grouped as 'Other'):")
for m in rare_models:
    print(f"  {m}: {model_counts[m]} transactions")

# Create grouped column
# Keep Terrace separate — these are rare HDB landed properties (20 transactions)
# that behave completely differently from other flat types
df['flat_model_grouped'] = df['flat_model'].apply(
    lambda x: x if x == 'Terrace' else ('Other' if x in rare_models else x))

print(f"\n=== flat_model_grouped counts (after grouping) ===")
print(df['flat_model_grouped'].value_counts())
print(f"\nTotal unique grouped models: {df['flat_model_grouped'].nunique()}")

=== flat_model counts (before grouping) ===
flat_model
Model A                   20121
Improved                  12017
New Generation             5798
Premium Apartment          4998
Simplified                 1899
Apartment                  1564
Standard                   1233
Maisonette                 1199
DBSS                        661
Model A2                    538
2-room                      323
Model A-Maisonette           77
Adjoined flat                75
Type S1                      74
3Gen                         46
Type S2                      28
Premium Apartment Loft       28
Terrace                      20
Multi Generation              7
Improved-Maisonette           6
Premium Maisonette            6
Name: count, dtype: int64

Total unique flat models: 21

Models with < 50 transactions (will be grouped as 'Other'):
  3Gen: 46 transactions
  Type S2: 28 transactions
  Premium Apartment Loft: 28 transactions
  Terrace: 20 transactions
  Multi Generation: 7 transactions
 

## Null check

Any nulls here would cause silent row-dropping in regression.

In [6]:
print("=== Null check ===")
null_counts = df.isnull().sum()
if null_counts.sum() > 0:
    print("WARNING: Nulls found in the following columns:")
    print(null_counts[null_counts > 0])
    print(f"\nTotal rows with any null: {df.isnull().any(axis=1).sum():,}")
else:
    print("No nulls in any column. Good to go.")

print(f"\nFinal shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

=== Null check ===


No nulls in any column. Good to go.

Final shape: 50,718 rows x 62 columns


## Save analysis-ready dataset

In [7]:
df.to_csv('data/hdb_analysis.csv', index=False)
print(f"Saved to data/hdb_analysis.csv ({df.shape[0]:,} rows, {df.shape[1]} cols)")
print(f"Columns: {df.columns.tolist()}")

# Quick sanity check: re-read and compare
df_check = pd.read_csv('data/hdb_analysis.csv')
assert df_check.shape == df.shape, f"Shape mismatch: {df_check.shape} vs {df.shape}"
print(f"\nSanity check passed: re-read file has same shape {df_check.shape}")

Saved to data/hdb_analysis.csv (50,718 rows, 62 cols)
Columns: ['month', 'town', 'block', 'street_name', 'resale_price', 'log_resale_price', 'flat_type', 'flat_model', 'floor_area_sqm', 'storey_range', 'storey_mid', 'lease_commence_date', 'remaining_lease_years', 'remaining_lease', 'ends_in_8', 'last_digit', 'dataset', 'latitude', 'longitude', 'dist_cbd_km', 'mrt_dist_m', 'nearest_mrt_name', 'school_dist_m', 'nearest_school_name', 'hawker_dist_m', 'nearest_hawker_name', 'supermarket_dist_m', 'nearest_supermarket_name', 'park_dist_m', 'nearest_park_name', 'columbarium_dist_m', 'nearest_columbarium_name', 'funeral_dist_m', 'nearest_funeral_name', 'hospital_dist_m', 'nearest_hospital_name', 'temple_dist_m', 'nearest_temple_name', 'reservoir_dist_m', 'nearest_reservoir_name', 'coast_dist_m', 'nearest_coast_name', 'popular_school_dist_m', 'nearest_popular_school_name', 'num_eights_in_price', 'num_fours_in_price', 'price_has_888', 'price_has_444', 'price_has_168', 'price_has_138', 'price_has

## Summary

`data/hdb_analysis.csv` is the analysis-ready file: original HDB variables (price, location, flat type, size, floor, lease), OneMap proximity distances (CBD, MRT, school, hawker, supermarket, park, hospital, temple, columbarium, funeral parlour, reservoir, coast), superstition variables (price digits, floor numbers, block numbers, Hungry Ghost Month, CNY), and `flat_model_grouped` with rare models collapsed into "Other".